# 10 - Unity Catalog privileges, row filters, masks, and ABAC

Prerequisites / execution context: Run 00. Creates a separate security_demo table. SQL grant templates require a real account group and suitable ownership; multi-user verification needs a second identity. Most admin operations are opt-in.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R58: UC privileges reference](https://docs.databricks.com/aws/en/data-governance/unity-catalog/access-control/privileges-reference); [R59: UC managed tables](https://docs.databricks.com/aws/en/tables/managed); [R60: External tables](https://docs.databricks.com/aws/en/tables/external); [R61: Convert to managed tables](https://docs.databricks.com/aws/en/tables/convert-to-managed); [R62: Table-level row filters and masks](https://docs.databricks.com/aws/en/data-governance/unity-catalog/filters-and-masks/); [R63: Apply row filters and masks](https://docs.databricks.com/aws/en/data-governance/unity-catalog/filters-and-masks/manually-apply); [R64: ABAC overview](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/); [R65: ABAC policy creation](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/policies); [R66: ABAC DENY policies (Beta)](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/deny-policies); [R67: Unity Catalog lineage](https://docs.databricks.com/aws/en/data-governance/unity-catalog/data-lineage).


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


## 1. Inspect metadata
Managed and external are storage lifecycle choices; both can be governed by UC. Do not create an external table inside a volume or use a managed table path as an external location.


In [ ]:
T=f"{SILVER}.security_demo"
spark.sql(f"CREATE OR REPLACE TABLE {T} (id INT,country STRING,email STRING) USING DELTA")
spark.sql(f"INSERT INTO {T} VALUES (1,'IT','one@example.test'),(2,'FR','two@example.test')")
spark.sql(f"DESCRIBE EXTENDED {T}").show(truncate=False)
spark.sql(f"SHOW GRANTS ON TABLE {T}").show(truncate=False)


## 2. Least-privilege grant and revoke
Set GROUP to an existing account group. SQL templates are printed so you can inspect the scope before applying. Revoking a direct grant does not remove access inherited through another group or parent object. Official Databricks documentation: [R58: UC privileges reference](https://docs.databricks.com/aws/en/data-governance/unity-catalog/access-control/privileges-reference).


In [ ]:
GROUP="dea_readers"  # existing account group
print(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{GROUP}`;")
print(f"GRANT USE SCHEMA ON SCHEMA {SILVER} TO `{GROUP}`;")
print(f"GRANT SELECT ON TABLE {T} TO `{GROUP}`;")
print(f"REVOKE SELECT ON TABLE {T} FROM `{GROUP}`;")


## 3. Apply policies on the disposable demo
Opt in only if the finance_team account group exists and you own the demo table. Other users see IT rows only and masked email; finance members see both countries and full email. The function parameter type matches the bound column type. Official Databricks documentation: [R63: Apply row filters and masks](https://docs.databricks.com/aws/en/data-governance/unity-catalog/filters-and-masks/manually-apply).


In [ ]:
APPLY_POLICIES=False
if APPLY_POLICIES:
    spark.sql(f"""CREATE OR REPLACE FUNCTION {SILVER}.kit_email_mask(v STRING)
    RETURNS STRING RETURN CASE WHEN is_account_group_member('finance_team') THEN v ELSE 'REDACTED' END""")
    spark.sql(f"""CREATE OR REPLACE FUNCTION {SILVER}.kit_region_filter(v STRING)
    RETURNS BOOLEAN RETURN is_account_group_member('finance_team') OR v='IT'""")
    spark.sql(f"ALTER TABLE {T} ALTER COLUMN email SET MASK {SILVER}.kit_email_mask")
    spark.sql(f"ALTER TABLE {T} SET ROW FILTER {SILVER}.kit_region_filter ON (country)")
    spark.table(T).show()


## 4. Prove access using identities
As an authorized group member, read the demo. Repeat as a non-member with the required table/parent privileges. Record row count and email visibility. Do not claim security is tested by querying once as the owner. Remove table-level bindings before dropping their functions: ALTER TABLE ... DROP ROW FILTER; ALTER TABLE ... ALTER COLUMN email DROP MASK. Official Databricks documentation: [R63: Apply row filters and masks](https://docs.databricks.com/aws/en/data-governance/unity-catalog/filters-and-masks/manually-apply).


## 5. ABAC guided exercise
In a permitted development schema, create a governed tag for synthetic sensitive columns, create a masking SQL UDF, and use Catalog Explorer Policies to create a column-mask policy scoped to tagged columns for your reader group. Use Show code and compare it with R65. Tag email on two new demo tables; verify that both receive the policy. Remove a tag and re-evaluate the effective policy. Record who can assign the tag and modify the policy. Keep this separate from the manually masked table to avoid overlapping-policy ambiguity. If tag/policy administration is unavailable, submit the design and test matrix as a paper exercise. Official Databricks documentation: [R64: ABAC overview](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/); [R65: ABAC policy creation](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/policies).


## 6. DENY distinction
The exam outline mentions DENY. Current official ABAC DENY is Beta, restricts MANAGE ACCESS CONTROL, and has its own requirements. Do not translate it into a generic DENY SELECT command or use it as column masking. Read the current supported privilege list before applying. Official Databricks documentation: [R66: ABAC DENY policies (Beta)](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/deny-policies).


## 7. Conversion and lineage
Optional admin exercise: on a disposable external Delta table under an authorized external location, execute ALTER TABLE ... SET MANAGED, inspect its type and location, then ALTER TABLE ... UNSET MANAGED within the documented 14-day rollback window. This is rollback of that conversion, not an arbitrary managed-to-external conversion. Use R61 for runtime and access requirements. Finally inspect the main Gold table lineage in Catalog Explorer. Official Databricks documentation: [R61: Convert to managed tables](https://docs.databricks.com/aws/en/tables/convert-to-managed); [R67: Unity Catalog lineage](https://docs.databricks.com/aws/en/data-governance/unity-catalog/data-lineage).
